In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
ROOT_DIR = Path('/projects/ai_science_alphafold')
OUT_DIR = ROOT_DIR / 'processed'

run_config = pd.read_json(OUT_DIR / 'run_config.json').iloc[0].to_dict()
author_panel = pd.read_csv(OUT_DIR / 'author_panel.csv')

EXPORT_TO_CSV = False
outpath = Path('/projects/prize_winner/ai_matthew_science/reg/data') / (
    f'{int(run_config["MATCHING_FIRST_YEAR"])}-{int(run_config["MATCHING_LAST_YEAR"])}_'
    f'{run_config["AFF_PREFIX"]}{int(run_config["TOP_AFF_RANK"])}.csv'
)

print('Rows:', len(author_panel), 'Columns:', len(author_panel.columns))
print('Export target:', outpath)

In [ ]:
export_df = author_panel.copy()
export_df = export_df[export_df['pub_count'] >= 5]
export_df['treated'] = export_df['treated'].astype('int8')
export_df['is_top_aff'] = export_df['is_top_aff'].astype('int8')
export_df.columns = [c.replace('_Year_', '_') for c in export_df.columns]

if EXPORT_TO_CSV:
    outpath.parent.mkdir(parents=True, exist_ok=True)
    export_df.to_csv(outpath, index=False)

meta = {
    'row_count': int(export_df.shape[0]),
    'column_count': int(export_df.shape[1]),
    'export_path': str(outpath),
    'did_write_csv': bool(EXPORT_TO_CSV),
    'config': run_config,
}
with open(OUT_DIR / 'export_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

export_df.head()

In [ ]:
summary = export_df.groupby('treated', dropna=False).agg(
    n_authors=('author_id', 'nunique'),
    mean_pub_count=('pub_count', 'mean'),
)
summary